# 06 Learned methods on the test cohort (H2, H3)
Every trained network is applied to the clinical-grade protocols of the test subjects, then tracked and scored exactly like the classical methods; the last cell computes the path error of classical CSD and every network against the reference fODF.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
import copy, torch
from retinopath.train import load_trained, net_fodf_all, trained_runs
subjects = PL.make_subject_lists(cfg); test = [PL.session(sid) for sid in subjects["test"]]
clinical = [protocol for protocol in cfg.clinical_protocols if protocol in cfg.run_protocols]; runs = trained_runs(cfg); print("trained runs:", runs)
with Timer(cfg, "06_inference"):
    for variant, seed in runs:
        variant_cfg = copy.deepcopy(cfg); model, meta = load_trained(variant_cfg, variant, seed, device)
        net_fodf_all(variant_cfg, test, clinical, f"net_{variant}_s{seed}", model, device)

In [ ]:
with Timer(cfg, "06_tracking_scores"):
    scores = PL.run_grid(cfg, test, [(protocol, f"net_{variant}_s{seed}") for protocol in clinical for variant, seed in runs], "learned")
scores["variant"] = scores.method.str.extract(r"net_(.*)_s\d+")[0]
scores.groupby(["protocol", "variant"])[["n_pairs", "rsi_ree_all", "acc", "peak_err_deg", "wdice_ref"]].median().round(3)

In [ ]:
# Path error: distance between paths traced through each fODF and through the reference fODF (tracer of the training loss)
from retinopath.train import path_errors
with Timer(cfg, "06_path_error"):
    path_error = path_errors(cfg, test, [(protocol, method) for protocol in clinical for method in ["csd"] + [f"net_{variant}_s{seed}" for variant, seed in runs]], "path_error", device)
path_error["variant"] = path_error.method.str.extract(r"net_(.*)_s\d+")[0].fillna(path_error.method)
path_error.groupby(["protocol", "variant"])[["path_err_mm", "path_end_err_mm"]].median().round(3)